In [1]:
import numpy as np
import pandas as pd
import os, sys, importlib
package_path = os.path.abspath('../')
if package_path not in sys.path:
    sys.path.append(package_path)
from package import functions as fn
from obspy import read
from obspy import Stream
from collections import defaultdict
import copy
from matplotlib import pyplot as plt
from scipy.fft import fft, fftfreq
from obspy.clients.fdsn import Client
from obspy.geodetics import gps2dist_azimuth
from obspy.geodetics import kilometers2degrees
from obspy.taup import TauPyModel
import pickle
from obspy import Stream

In [2]:
# Latitude 	Longitude 	Date 	Depth 	Magnitude 	Description
# 18.820333° N 	155.527167° W 	2021-10-10 21:48:36 UTC 	35.06 km 	Ml6.2 	Hawaii

catalog_columns = ['event_id', 'time', 'latitude', 'longitude', 'depth']
catalog_df = pd.DataFrame(columns=catalog_columns)
event_id = 1
time = '2021-10-10 21:48:36'
eq_lat = 18.820333
eq_lon = -155.527167
hdepth = 35.06
catalog_df.loc[len(catalog_df)] = [event_id, time, eq_lat, eq_lon, hdepth]
catalog_df

# save to csv
if not os.path.exists('catalog.csv'):
    catalog_df.to_csv('catalog.csv', index=False)

In [3]:
# get the waveforms downloaded from IRIS
data_path = '../data/2021-10-10-ml62-hawaii-all'
filenames = [f for f in os.listdir(data_path) if f.endswith('.SAC')]
station_names = {f.split('.')[1] for f in filenames}
filt_filenames = [f for f in filenames if f.split('.')[1] in station_names]
print(f"Number of stations: {len(station_names)}")

Number of stations: 0


In [ ]:
streams = [read(data_path + '/' + f) for f in filt_filenames]
traces = [st[0] for st in streams if len(st) == 1]
assert len(traces) == len(filt_filenames), "Some files contain more than one trace."
order = ['BH1', 'BH2', 'BHE', 'BHN', 'BHZ']
grouped_traces = defaultdict(list)
for tr in traces:
    grouped_traces[tr.stats.station].append(tr)
for station, tr_list in grouped_traces.items():
    tr_list.sort(key=lambda tr: order.index(tr.stats.channel))
grouped_streams = {station: Stream(traces=tr_list)
    for station, tr_list in grouped_traces.items()}

client = Client("IRIS")
vel_model = TauPyModel(model="ak135")

###################################

az_epdist = defaultdict(list)
p_to_arvs_inc = defaultdict(list)
s_to_arvs_inc = defaultdict(list)   
incs = {}
inventories = {}
z_streams = {}
l_streams_p = {}
l_streams_s = {}
count = 0
for station, st in grouped_streams.items():   
    # if station != 'ANMO': continue
    # I wanna rotate all the streams to LQT, I need epdist which comes from sta_lat and sta_lon
    sta_lat = st[0].stats.sac.stla
    sta_lon = st[0].stats.sac.stlo
    epdist, az_alt, baz_alt = gps2dist_azimuth(eq_lat, eq_lon, sta_lat, sta_lon)
    epdist_deg = kilometers2degrees(epdist / 1000.)
    az = st[0].stats.sac.az
    baz = st[0].stats.sac.baz
    p_arrivals = vel_model.get_travel_times(source_depth_in_km=hdepth,
                                            distance_in_degree=epdist_deg,
                                            phase_list=["P", "p", "Pn", "Pg", "pP", "sP"])
    if len(p_arrivals) == 0:
        print(f"No P arrivals found for station {station}. Skipping.")
        continue
    p_arrival = min(p_arrivals, key=lambda x: x.time)
    p_inc = p_arrival.incident_angle
    p_to = 180 - p_arrival.takeoff_angle
    s_arrivals = vel_model.get_travel_times(source_depth_in_km=hdepth,
                                            distance_in_degree=epdist_deg,   
                                            phase_list=["S", "s", "Sn", "Sg", "sS", "pS"])
    if len(s_arrivals) == 0:
        print(f"No S arrivals found for station {station}. Skipping.")
        continue
    s_arrival = min(s_arrivals, key=lambda x: x.time)
    s_inc = s_arrival.incident_angle
    s_to = 180 - s_arrival.takeoff_angle
    az_epdist[station].append(az)
    az_epdist[station].append(epdist_deg)
    p_to_arvs_inc[station].append(p_to)
    p_to_arvs_inc[station].append(p_arrivals)
    p_to_arvs_inc[station].append(p_inc)
    s_to_arvs_inc[station].append(s_to)
    s_to_arvs_inc[station].append(s_arrivals)
    s_to_arvs_inc[station].append(s_inc)
    network = st[0].stats.network
    starttime = st[0].stats.starttime
    endtime = st[0].stats.endtime
    try:
        inv = client.get_stations(network=network, station=station, starttime=starttime,
                                endtime=endtime, level="response")
    except Exception as e:
        print(f"Error fetching inventory for station {station}: {e}")
        continue
    inventories[station] = inv
    for tr in st: # NOTE: pay attention to the frequency range
        tr.remove_response(inventory=inv, output="DISP", pre_filt=(0.01, 0.02, 8.0, 10.0),
                           water_level=None, plot=False)
    st._rotate_to_zne(inventory=inv)
    try:
        stZ = copy.deepcopy(st)
        stLp = copy.deepcopy(st)
        stLs = copy.deepcopy(st)
        stZ.rotate('NE->RT', back_azimuth=baz)
        stLp.rotate('ZNE->LQT', back_azimuth=baz, inclination=p_inc)
        stLs.rotate('ZNE->LQT', back_azimuth=baz, inclination=s_inc)
        stLp[1].data = -stLp[1].data  # invert the Qp->O component
        stLs[1].data = -stLs[1].data  # invert the Qs->O component
    except Exception as e:
        print(f"Error rotating stream for station {station}: {e}")
        continue
    z_streams[station] = stZ
    l_streams_p[station] = stLp
    l_streams_s[station] = stLs
    count += 1
    if count % 10 == 0:
        print(f"Count: {count} | Station: {station}")
    # if count >= 1: break

# order the stations in l_streams, z_streams, az_to_arvs_epdist_inc, and inventories by epdist
sorted_stations = sorted(z_streams.keys(), key=lambda s: az_epdist[s][1])
z_streams = {station: z_streams[station] for station in sorted_stations}
l_streams_p = {station: l_streams_p[station] for station in sorted_stations}
l_streams_s = {station: l_streams_s[station] for station in sorted_stations}
az_epdist = {station: az_epdist[station] for station in sorted_stations}
p_to_arvs_inc = {station: p_to_arvs_inc[station] for station in sorted_stations}
s_to_arvs_inc = {station: s_to_arvs_inc[station] for station in sorted_stations}
inventories = {station: inventories[station] for station in sorted_stations}


with open("processed_data.pkl", "wb") as f:
    pickle.dump((z_streams, l_streams_p, l_streams_s, az_epdist, p_to_arvs_inc,
                 s_to_arvs_inc, inventories), f)

##################################
